In [1]:
#import statements
import pandas as pd
import matplotlib.pyplot as plt
import subprocess
import numpy as np
import nilearn.image
import nilearn.plotting
import tempfile
import subprocess, shutil
import numpy as np
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import copy
import shutil
from torch.utils.data import random_split
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from pathlib import Path
import ants
import pydicom
import nibabel as nib
import os
from glob import glob
from tqdm import tqdm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torchinfo import summary
import nilearn

os.environ.setdefault("FSLDIR", "/Users/william.wakefield/fsl")
fsl_bin = f"{os.environ['FSLDIR']}/share/fsl/bin"
if fsl_bin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = fsl_bin + os.pathsep + os.environ.get("PATH", "")
os.environ.setdefault("FSLOUTPUTTYPE", "NIFTI_GZ")

'NIFTI_GZ'

In [5]:
vbm_root = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm")
raw_dir = vbm_root / "t1_long_raw"
strip_dir = vbm_root / "t1_long_skull_strip"

raw_files = sorted(raw_dir.glob("*.nii"))
print(f"Found {len(raw_files)} raw native T1 images to skull-strip")

Found 802 raw native T1 images to skull-strip


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

raw_files = sorted(raw_dir.glob("*.nii"))
print(f"Skull-strip {len(raw_files)} T1 volumes (bet -f 0.4 -g 0 -B)")


def _run_bet(raw_path, out_nii):
    result = subprocess.run(
        ["bet", str(raw_path), str(out_nii), "-f", "0.4", "-g", "0", "-B"],
        capture_output=True, text=True,
    )
    return raw_path.stem, result.returncode, result.stderr.strip()

failed = []
skipped = 0
tasks = []

for raw_path in raw_files:
    out_nii = strip_dir / f"{raw_path.stem}.nii.gz"
    if out_nii.exists():
        skipped += 1
        continue
    tasks.append((raw_path, out_nii))

bet_workers = int(os.environ["BET_MAX_WORKERS"]) if os.environ.get("BET_MAX_WORKERS") else (os.cpu_count() or 8)
max_workers = max(1, min(len(tasks) or 1, bet_workers))
print(f"BET: {len(tasks)} jobs, {max_workers} workers ({skipped} already done)")

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(_run_bet, raw, out) for raw, out in tasks]
    for fut in tqdm(as_completed(futures), total=len(tasks), desc="BET native T1"):
        stem, rc, err = fut.result()
        if rc != 0:
            failed.append((stem, err))

stripped = [f for f in strip_dir.glob("*.nii.gz") if not f.name.endswith("_mask.nii.gz")]
print(f"\nNative skull stripping complete: {len(stripped)} images")
print(f"  ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

In [6]:
vbm_root = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm")
strip_dir = vbm_root / "t1_long_skull_strip"
seg_dir = vbm_root / "t1_long_seg_native"

stripped_files = sorted(f for f in strip_dir.glob("*.nii.gz") if not f.name.endswith("_mask.nii.gz"))
print(f"Found {len(stripped_files)} native skull-stripped T1 images to segment")

Found 802 native skull-stripped T1 images to segment


In [8]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def _run_fast(strip_path, out_base):
    result = subprocess.run(
        [
            "fast",
            "-t", "1",
            "-n", "3",
            "-B",
            "-b",
            "-o", str(out_base),
            str(strip_path),
        ],
        capture_output=True, text=True,
    )
    stem = strip_path.name.replace(".nii.gz", "")
    return stem, result.returncode, result.stderr.strip()

failed = []
skipped = 0
tasks = []

for strip_path in stripped_files:
    stem = strip_path.name.replace(".nii.gz", "")
    subject_dir = seg_dir / stem
    subject_dir.mkdir(parents=True, exist_ok=True)
    out_base = subject_dir / stem

    pve_csf = subject_dir / f"{stem}_pve_0.nii.gz"
    pve_gm = subject_dir / f"{stem}_pve_1.nii.gz"
    pve_wm = subject_dir / f"{stem}_pve_2.nii.gz"

    if pve_csf.exists() and pve_gm.exists() and pve_wm.exists():
        skipped += 1
        continue

    tasks.append((strip_path, out_base))

max_workers = max(1, (os.cpu_count() or 4) - 2)
print(f"Running {len(tasks)} FASTs in parallel with {max_workers} workers "
      f"({skipped} skipped, already existed)")

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(_run_fast, sp, ob) for sp, ob in tasks]
    for fut in tqdm(as_completed(futures), total=len(tasks), desc="FAST native T1"):
        stem, rc, err = fut.result()
        if rc != 0:
            failed.append((stem, err))

csf_maps = list(seg_dir.glob("*/*_pve_0.nii.gz"))
gm_maps = list(seg_dir.glob("*/*_pve_1.nii.gz"))
wm_maps = list(seg_dir.glob("*/*_pve_2.nii.gz"))
print(f"\nNative FAST segmentation complete:")
print(f"  {len(csf_maps)} CSF maps (pve_0)")
print(f"  {len(gm_maps)} GM maps (pve_1)")
print(f"  {len(wm_maps)} WM maps (pve_2)")
print(f"  ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Running 802 FASTs in parallel with 14 workers (0 skipped, already existed)


FAST native T1: 100%|██████████| 802/802 [2:33:49<00:00, 11.51s/it]  


Native FAST segmentation complete:
  802 CSF maps (pve_0)
  802 GM maps (pve_1)
  802 WM maps (pve_2)
  (0 skipped, already existed)


In [ ]:
## ants syn skull stripped t1 to skull stripped mni

In [7]:
vbm_root = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm")
bet_dir = vbm_root / "t1_long_skull_strip"
warps_dir = vbm_root / "t1_long_vbm_warps"
warps_dir.mkdir(parents=True, exist_ok=True)

mni_t1_path = Path("model_data/mni_t1_stripped.nii")
mni_t1 = ants.image_read(str(mni_t1_path))

bet_files = sorted(f for f in bet_dir.glob("*.nii.gz") if not f.name.endswith("_mask.nii.gz"))
print(f"Found {len(bet_files)} skull stripped native T1 images to register")

Found 802 skull stripped native T1 images to register


In [10]:
failed = []
skipped = 0

for bet_path in tqdm(bet_files, desc="SyN bet T1 -> MNI"):
    stem = bet_path.stem
    out_warped = warps_dir / f"{stem}_warpedT1.nii.gz"
    out_affine = warps_dir / f"{stem}_0GenericAffine.mat"
    out_warp = warps_dir / f"{stem}_1Warp.nii.gz"
    out_inv_warp = warps_dir / f"{stem}_1InverseWarp.nii.gz"

    if (
        out_warped.exists()
        and out_affine.exists()
        and out_warp.exists()
        and out_inv_warp.exists()
    ):
        skipped += 1
        continue

    try:
        moving = ants.image_read(str(bet_path))
        reg = ants.registration(fixed=mni_t1, moving=moving, type_of_transform="SyN")

        # Warp original (unnormalized) moving to preserve native T1 intensities
        warped_native = ants.apply_transforms(
            fixed=mni_t1, moving=moving,
            transformlist=reg["fwdtransforms"], interpolator="linear")
        ants.image_write(warped_native, str(out_warped))

        for tx_path in reg["fwdtransforms"]:
            tx = Path(tx_path)
            if "GenericAffine" in tx.name:
                shutil.copy(tx, out_affine)
            elif "Warp" in tx.name and "Inverse" not in tx.name:
                shutil.copy(tx, out_warp)

        for tx_path in reg["invtransforms"]:
            tx = Path(tx_path)
            if "InverseWarp" in tx.name:
                shutil.copy(tx, out_inv_warp)

    except Exception as e:
        failed.append((stem, str(e)))

warped = list(warps_dir.glob("*_warpedT1.nii.gz"))
affines = list(warps_dir.glob("*_0GenericAffine.mat"))
fwd_warps = list(warps_dir.glob("*_1Warp.nii.gz"))
inv_warps = list(warps_dir.glob("*_1InverseWarp.nii.gz"))
print(f"\nVBM SyN registration complete:")
print(f"  {len(warped)} warped T1s")
print(f"  {len(affines)} affines, {len(fwd_warps)} forward warps, {len(inv_warps)} inverse warps")
print(f"  ({skipped} skipped, already existed)")
if failed:
    print(f"{len(failed)} failures:")
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

SyN bet T1 -> MNI: 100%|██████████| 802/802 [2:04:21<00:00,  9.30s/it]  


VBM SyN registration complete:
  802 warped T1s
  802 affines, 802 forward warps, 802 inverse warps
  (0 skipped, already existed)


In [16]:
vbm_root     = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm")
jac_dir      = vbm_root / "t1_long_jacobians"
pve_mni_dir  = vbm_root / "t1_long_pve_mni"
pve_mod_dir  = vbm_root / "t1_long_pve_modulated"
pve_smooth_dir = vbm_root / "t1_long_pve_smoothed"
for d in (jac_dir, pve_mni_dir, pve_mod_dir, pve_smooth_dir):
    d.mkdir(parents=True, exist_ok=True)

warp_files = sorted(warps_dir.glob("*_1Warp.nii.gz"))
print(f"Jacobians / PVE warp: {len(warp_files)} subjects")

Jacobians / PVE warp: 802 subjects


In [17]:
# --- Jacobian determinant (SyN warp × affine det) ----------------------------
failed, skipped = [], 0
for warp_path in tqdm(warp_files, desc="Jacobian"):
    stem = warp_path.name.replace("_1Warp.nii.gz", "")
    affine_path = warps_dir / f"{stem}_0GenericAffine.mat"
    out_jac = jac_dir / f"{stem}_jacobian.nii.gz"
    if out_jac.exists():
        skipped += 1; continue
    if not affine_path.exists():
        failed.append((stem, "missing affine")); continue
    try:
        jac_warp = ants.create_jacobian_determinant_image(
            domain_image=mni_t1, tx=str(warp_path), do_log=False, geom=False)
        params = np.asarray(ants.read_transform(str(affine_path)).parameters)
        det_affine = abs(float(np.linalg.det(params[:9].reshape(3, 3))))
        ants.image_write(jac_warp * det_affine, str(out_jac))
    except Exception as e:
        failed.append((stem, str(e)))

print(f"Jacobian: {len(list(jac_dir.glob('*_jacobian.nii.gz')))} | skipped {skipped} | failed {len(failed)}")
for s, e in failed[:5]: print(f"  {s}: {e}")

Jacobian: 100%|██████████| 802/802 [09:35<00:00,  1.39it/s]

Jacobian: 802 | skipped 0 | failed 0


In [18]:
# --- Warp native PVEs → MNI ---------------------------------------------------
# Seg dir stems omit ".nii"; warp files include it (bet_path.stem on .nii.gz).
failed, skipped = [], 0
subject_dirs = sorted(p for p in seg_dir.iterdir() if p.is_dir())
for subj_dir in tqdm(subject_dirs, desc="PVE→MNI"):
    stem = subj_dir.name
    out_sub = pve_mni_dir / stem
    out_sub.mkdir(parents=True, exist_ok=True)
    warp_stem = f"{stem}.nii"
    affine = warps_dir / f"{warp_stem}_0GenericAffine.mat"
    warp   = warps_dir / f"{warp_stem}_1Warp.nii.gz"
    if not affine.exists() or not warp.exists():
        failed.append((stem, "missing transforms")); continue
    if all((out_sub / f"{stem}_pve_{i}_mni.nii.gz").exists() for i in range(3)):
        skipped += 1; continue
    try:
        for i in range(3):
            out_pve = out_sub / f"{stem}_pve_{i}_mni.nii.gz"
            if out_pve.exists(): continue
            in_pve = subj_dir / f"{stem}_pve_{i}.nii.gz"
            if not in_pve.exists():
                failed.append((stem, f"missing pve_{i}")); break
            warped = ants.apply_transforms(
                fixed=mni_t1, moving=ants.image_read(str(in_pve)),
                transformlist=[str(warp), str(affine)], interpolator="linear")
            ants.image_write(warped, str(out_pve))
    except Exception as e:
        failed.append((stem, str(e)))

print(f"PVE→MNI: {len(list(pve_mni_dir.iterdir()))} dirs | skipped {skipped} | failed {len(failed)}")
for s, e in failed[:5]: print(f"  {s}: {e}")

PVE→MNI: 100%|██████████| 802/802 [28:02<00:00,  2.10s/it]  

PVE→MNI: 803 dirs | skipped 0 | failed 0


In [19]:
# --- Modulate MNI PVE × Jacobian ---------------------------------------------
failed, skipped = [], 0
for subj_dir in tqdm(sorted(p for p in pve_mni_dir.iterdir() if p.is_dir()), desc="Modulate"):
    stem = subj_dir.name
    out_sub = pve_mod_dir / stem
    out_sub.mkdir(parents=True, exist_ok=True)
    jac_path = jac_dir / f"{stem}.nii_jacobian.nii.gz"
    if not jac_path.exists():
        failed.append((stem, "missing jacobian")); continue
    if all((out_sub / f"{stem}_pve_{i}_mod.nii.gz").exists() for i in range(3)):
        skipped += 1; continue
    try:
        jac_arr = ants.image_read(str(jac_path)).numpy()
        for i in range(3):
            out_pve = out_sub / f"{stem}_pve_{i}_mod.nii.gz"
            if out_pve.exists(): continue
            in_pve = subj_dir / f"{stem}_pve_{i}_mni.nii.gz"
            if not in_pve.exists():
                failed.append((stem, f"missing pve_{i}_mni")); continue
            pve_img = ants.image_read(str(in_pve))
            mod = ants.from_numpy(pve_img.numpy() * jac_arr,
                                  origin=pve_img.origin, spacing=pve_img.spacing,
                                  direction=pve_img.direction)
            ants.image_write(mod, str(out_pve))
    except Exception as e:
        failed.append((stem, str(e)))

print(f"Modulated: {len(list(pve_mod_dir.glob('*/*_pve_0_mod.nii.gz')))} | skipped {skipped} | failed {len(failed)}")
for s, e in failed[:5]: print(f"  {s}: {e}")

Modulate: 100%|██████████| 802/802 [06:06<00:00,  2.19it/s]

Modulated: 802 | skipped 0 | failed 0


In [20]:
# --- Smooth modulated PVEs (2.5 mm FWHM) -------------------------------------
fwhm_mm  = 2.5
sigma_mm = fwhm_mm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
failed, skipped = [], 0
for subj_dir in tqdm(sorted(p for p in pve_mod_dir.iterdir() if p.is_dir()), desc="Smooth"):
    stem = subj_dir.name
    out_sub = pve_smooth_dir / stem
    out_sub.mkdir(parents=True, exist_ok=True)
    if all((out_sub / f"{stem}_pve_{i}_mod_s2p5.nii.gz").exists() for i in range(3)):
        skipped += 1; continue
    try:
        for i in range(3):
            out_pve = out_sub / f"{stem}_pve_{i}_mod_s2p5.nii.gz"
            if out_pve.exists(): continue
            in_pve = subj_dir / f"{stem}_pve_{i}_mod.nii.gz"
            if not in_pve.exists():
                failed.append((stem, f"missing pve_{i}_mod")); continue
            sm = ants.smooth_image(ants.image_read(str(in_pve)),
                                   sigma=sigma_mm, sigma_in_physical_coordinates=True)
            ants.image_write(sm, str(out_pve))
    except Exception as e:
        failed.append((stem, str(e)))

print(f"Smoothed: {len(list(pve_smooth_dir.glob('*/*_pve_1_mod_s2p5.nii.gz')))} GM | skipped {skipped} | failed {len(failed)}")
for s, e in failed[:5]: print(f"  {s}: {e}")

Smooth: 100%|██████████| 802/802 [07:51<00:00,  1.70it/s]

Smoothed: 802 GM | skipped 0 | failed 0


In [ ]:
# ── eddy_cpu: motion & eddy-current correction ────────────────────────────
# Inputs:  dti_long_raw/{stem}.nii.gz + .bval + .bvec + .json
#          dti_long_bet/{stem}_mask.nii.gz
# Outputs: dti_long_eddy/{stem}.nii.gz  (corrected 4-D DWI)
#          dti_long_eddy/{stem}.eddy_rotated_bvecs
# M3 Max: 4 parallel jobs × 4 threads each = 16 cores fully used.

import json as _json, tempfile, shutil
from concurrent.futures import ThreadPoolExecutor, as_completed

dti_vbm_root_eddy = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm")
dti_raw_dir  = Path("model_data/adni/dti_long_data/dti_long_raw")
dti_bet_dir  = dti_vbm_root_eddy / "dti_long_bet"
dti_eddy_dir = dti_vbm_root_eddy / "dti_long_eddy"

_ACQP_FALLBACK = "0 1 0 0.0448"   # A>>P, typical ADNI EPI readout time

def _acqparams_line(json_path: Path) -> str:
    try:
        meta = _json.loads(json_path.read_text())
        ped  = meta.get("PhaseEncodingDirection", "")
        trt  = float(meta.get("TotalReadoutTime", 0) or 0)
        if not ped or trt <= 0:
            return _ACQP_FALLBACK
        pe_vec = {"i": "1 0 0", "i-": "-1 0 0",
                  "j": "0 1 0", "j-":  "0 -1 0",
                  "k": "0 0 1", "k-":  "0 0 -1"}.get(ped, "0 1 0")
        return f"{pe_vec} {trt:.4f}"
    except Exception:
        return _ACQP_FALLBACK


def _run_eddy(stem: str) -> tuple[str, str]:
    nii_path  = dti_raw_dir  / f"{stem}.nii.gz"
    bval_path = dti_raw_dir  / f"{stem}.bval"
    bvec_path = dti_raw_dir  / f"{stem}.bvec"
    json_path = dti_raw_dir  / f"{stem}.json"
    mask_path = dti_bet_dir  / f"{stem}_mask.nii.gz"
    out_nii   = dti_eddy_dir / f"{stem}.nii.gz"
    out_bvec  = dti_eddy_dir / f"{stem}.eddy_rotated_bvecs"

    if out_nii.exists() and out_bvec.exists():
        return stem, "skip"
    for p in [nii_path, bval_path, bvec_path, mask_path]:
        if not p.exists():
            return stem, f"fail:missing_{p.name}"

    try:
        bvals  = np.loadtxt(str(bval_path)).flatten()
        n_vols = len(bvals)

        with tempfile.TemporaryDirectory() as _tmp:
            tmp        = Path(_tmp)
            acqp_file  = tmp / "acqparams.txt"
            index_file = tmp / "index.txt"
            out_prefix = tmp / stem

            acqp_file.write_text(_acqparams_line(json_path) + "\n")
            index_file.write_text(" ".join(["1"] * n_vols) + "\n")

            _env = {**os.environ, "OMP_NUM_THREADS": "4"}
            r = subprocess.run(
                [
                    "eddy_cpu",
                    f"--imain={nii_path}",
                    f"--mask={mask_path}",
                    f"--acqp={acqp_file}",
                    f"--index={index_file}",
                    f"--bvecs={bvec_path}",
                    f"--bvals={bval_path}",
                    f"--out={out_prefix}",
                    "--niter=3",
                ],
                capture_output=True, text=True, env=_env,
            )
            if r.returncode != 0:
                return stem, f"fail:eddy_rc{r.returncode} {r.stderr[:300]}"

            shutil.copy2(f"{out_prefix}.nii.gz", out_nii)
            shutil.copy2(f"{out_prefix}.eddy_rotated_bvecs", out_bvec)

        return stem, "ok"
    except Exception as e:
        return stem, f"fail:{e}"


eddy_stems = sorted(
    p.name.replace("_mask.nii.gz", "")
    for p in dti_bet_dir.glob("*_mask.nii.gz")
)
tasks     = [s for s in eddy_stems
             if not (dti_eddy_dir / f"{s}.nii.gz").exists()
             or not (dti_eddy_dir / f"{s}.eddy_rotated_bvecs").exists()]
skipped_n = len(eddy_stems) - len(tasks)

print(f"eddy_cpu: {len(tasks)} to run, {skipped_n} skipped (4 workers × 4 threads, --niter=3)")
failed = []
with ThreadPoolExecutor(max_workers=4) as ex:
    futures = {ex.submit(_run_eddy, s): s for s in tasks}
    for fut in tqdm(as_completed(futures), total=len(tasks), desc="eddy"):
        stem, status = fut.result()
        if "fail" in status:
            failed.append((stem, status))

eddy_nii = list(dti_eddy_dir.glob("*.nii.gz"))
print(f"\neddy complete: {len(eddy_nii)} corrected volumes | {len(failed)} failed")
if failed:
    for s, e in failed[:10]: print(f"  {s}: {e}")

eddy_cpu: 802 to run, 0 skipped (4 workers × 4 threads, --repol)


eddy:   0%|          | 0/802 [00:00<?, ?it/s]

In [ ]:
# ── dtifit ────────────────────────────────────────────────────────────────────
# Uses eddy-corrected 4-D DWI + rotated bvecs from dti_long_eddy.
dtifit_dir = dti_vbm_root_eddy / "dti_long_dtifit"
dtifit_dir.mkdir(parents=True, exist_ok=True)

EXCLUDE_STEMS = {"141_S_6116"}   # <6 DWI directions — tensor underdetermined

def check_bval_adequacy(bval_path: Path) -> tuple[bool, str]:
    bvals = np.loadtxt(str(bval_path)).flatten()
    n_b0  = int(np.sum(bvals < 50))
    n_dwi = int(np.sum(bvals >= 500))
    if n_b0 == 0:   return False, "no b=0 volumes"
    if n_dwi < 6:   return False, f"only {n_dwi} DWI directions — need ≥6"
    return True, f"ok ({n_b0} b=0, {n_dwi} DWI)"

def _run_dtifit(stem: str) -> tuple[str, str]:
    if stem in EXCLUDE_STEMS:
        return stem, "skip:excluded"

    nii_path  = dti_eddy_dir / f"{stem}.nii.gz"              # eddy-corrected
    bvec_path = dti_eddy_dir / f"{stem}.eddy_rotated_bvecs"  # rotated bvecs
    bval_path = dti_raw_dir  / f"{stem}.bval"
    mask_path = dti_bet_dir  / f"{stem}_mask.nii.gz"
    out_dir   = dtifit_dir   / stem
    out_md    = out_dir      / f"{stem}_MD.nii.gz"
    out_fa    = out_dir      / f"{stem}_FA.nii.gz"

    if out_md.exists() and out_fa.exists():
        return stem, "skip"
    for p in [nii_path, bval_path, bvec_path, mask_path]:
        if not p.exists():
            return stem, f"fail:missing_{p.name}"

    ok, msg = check_bval_adequacy(bval_path)
    if not ok:
        return stem, f"fail:bval_{msg}"

    out_dir.mkdir(parents=True, exist_ok=True)
    r = subprocess.run(
        [
            "dtifit",
            "-k", str(nii_path),
            "-o", str(out_dir / stem),
            "-m", str(mask_path),
            "-r", str(bvec_path),
            "-b", str(bval_path),
            "--sse",
        ],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        return stem, f"fail:dtifit_rc{r.returncode} {r.stderr[:200]}"
    return stem, "ok"

dti_stems = sorted(
    p.name.replace("_mask.nii.gz", "")
    for p in dti_bet_dir.glob("*_mask.nii.gz")
)
tasks     = [s for s in dti_stems
             if not (dtifit_dir / s / f"{s}_MD.nii.gz").exists()
             and s not in EXCLUDE_STEMS]
skipped_n = len(dti_stems) - len(tasks)

max_workers = max(1, (os.cpu_count() or 4) - 2)
print(f"dtifit: {len(tasks)} to run, {skipped_n} skipped, {max_workers} workers")
failed = []
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {ex.submit(_run_dtifit, s): s for s in tasks}
    for fut in tqdm(as_completed(futures), total=len(tasks), desc="dtifit"):
        stem, status = fut.result()
        if "fail" in status:
            failed.append((stem, status))

md_maps = list(dtifit_dir.glob("*/*_MD.nii.gz"))
fa_maps = list(dtifit_dir.glob("*/*_FA.nii.gz"))
print(f"\ndtifit complete: {len(md_maps)} MD maps | {len(fa_maps)} FA maps | {len(failed)} failed")
if failed:
    for s, e in failed[:10]: print(f"  {s}: {e}")

In [3]:
# --- DTI paths: reg_t1_native → MNI ------------------------------------------
# Depends on strip_dir and warps_dir defined in earlier cells.
import re

dti_vbm_root  = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm")
dtifit_dir    = dti_vbm_root / "dti_long_dtifit"
dti_strip_dir = dti_vbm_root / "dti_long_bet"
md_t1_dir     = dti_vbm_root / "dti_long_reg_t1_native"
md_mni_dir    = dti_vbm_root / "dti_long_mni"
md_smooth_dir = dti_vbm_root / "dti_long_mni_smoothed"

for d in (md_t1_dir, md_mni_dir, md_smooth_dir):
    d.mkdir(parents=True, exist_ok=True)

subj_pat = re.compile(r"^I\d+_(\d{3}_S_\d+)$")
md_maps  = sorted(dtifit_dir.glob("*/*_MD.nii.gz"))
print(f"Found {len(md_maps)} DTI MD maps in {dtifit_dir}")

Found 787 DTI MD maps in model_data/adni/dti_long_data/dti_long_modulated_vbm/dti_long_dtifit


In [6]:
# --- Match each DTI scan to the closest-date T1 for the same subject ----------
import pandas as pd

t1_dcm_root  = Path("model_data/adni/t1_long_data/t1_long_dcm")
dti_dcm_root = Path("model_data/adni/dti_long_data/dti_long_dcm")
_date_pat = re.compile(r"^(\d{4}-\d{2}-\d{2})")
_img_pat  = re.compile(r"^I?(\d+)$")


def _build_date_map(dcm_root):
    mapping = {}
    for subj_dir in dcm_root.iterdir():
        if not subj_dir.is_dir(): continue
        for mod_dir in subj_dir.iterdir():
            if not mod_dir.is_dir(): continue
            for date_dir in mod_dir.iterdir():
                if not date_dir.is_dir(): continue
                m = _date_pat.match(date_dir.name)
                if not m: continue
                scan_date = pd.Timestamp(m.group(1))
                for img_dir in date_dir.iterdir():
                    if not img_dir.is_dir(): continue
                    im = _img_pat.match(img_dir.name)
                    if im:
                        mapping[im.group(1)] = scan_date
    return mapping


def _stem_to_date(stem, date_map):
    img_id = stem.lstrip("I").split("_")[0]
    return date_map.get(img_id)


t1_date_raw  = _build_date_map(t1_dcm_root)
dti_date_raw = _build_date_map(dti_dcm_root)

t1_date  = {f.name.replace(".nii.gz", ""): _stem_to_date(f.name.replace(".nii.gz", ""), t1_date_raw)
            for f in sorted(strip_dir.glob("*.nii.gz")) if not f.name.endswith("_mask.nii.gz")}
dti_date = {subj_dir.name: _stem_to_date(subj_dir.name, dti_date_raw)
            for subj_dir in sorted(dtifit_dir.iterdir()) if subj_dir.is_dir()}

print(f"t1_date:  {sum(v is not None for v in t1_date.values())}/{len(t1_date)} resolved")
print(f"dti_date: {sum(v is not None for v in dti_date.values())}/{len(dti_date)} resolved")

t1_by_subj = {}
for f in sorted(f for f in strip_dir.glob("*.nii.gz") if not f.name.endswith("_mask.nii.gz")):
    stem = f.name.replace(".nii.gz", "")
    m = subj_pat.match(stem)
    if m:
        t1_by_subj.setdefault(m.group(1), []).append(stem)

dti_by_subj = {}
for md_path in md_maps:
    dti_stem = md_path.name.replace("_MD.nii.gz", "")
    m = subj_pat.match(dti_stem)
    if m:
        dti_by_subj.setdefault(m.group(1), []).append((dti_stem, md_path))

dti_to_t1 = {}
for subj, dti_list in dti_by_subj.items():
    t1_list = sorted(t1_by_subj.get(subj, []))
    if not t1_list:
        continue
    for dti_stem, _ in sorted(dti_list, key=lambda x: x[0]):
        dti_dt = dti_date.get(dti_stem)
        if dti_dt is None:
            idx = sorted(dti_list, key=lambda x: x[0]).index((dti_stem, _))
            dti_to_t1[dti_stem] = t1_list[min(idx, len(t1_list) - 1)]
            continue
        t1_with_dates = [(t, t1_date.get(t)) for t in t1_list]
        t1_with_dates = [(t, d) for t, d in t1_with_dates if d is not None]
        if not t1_with_dates:
            dti_to_t1[dti_stem] = t1_list[0]
        else:
            dti_to_t1[dti_stem] = min(t1_with_dates, key=lambda x: abs(x[1] - dti_dt))[0]

print(f"Matched {len(dti_to_t1)} DTI scans to T1 scans")

t1_date:  802/802 resolved
dti_date: 787/787 resolved
Matched 786 DTI scans to T1 scans


In [12]:
def _register_b0_and_warp_md(md_path):
    dti_stem = md_path.name.replace("_MD.nii.gz", "")
    t1_stem  = dti_to_t1.get(dti_stem)

    if t1_stem is None:
        return dti_stem, "fail", "no matching T1 for subject"

    out_md_t1  = md_t1_dir  / f"{dti_stem}_MD.nii.gz"
    out_md_mni = md_mni_dir / f"{dti_stem}_MD.nii.gz"
    out_tx     = md_t1_dir  / f"{dti_stem}_rigid.mat"

    if out_md_mni.exists() and out_tx.exists():
        return dti_stem, "skip", ""

    b0_path   = dti_strip_dir / f"{dti_stem}_meanb0.nii.gz"
    mask_path = dti_strip_dir / f"{dti_stem}_mask.nii.gz"
    t1_path   = strip_dir     / f"{t1_stem}.nii.gz"

    # SyN cell used bet_path.stem on a .nii.gz file -> Python gives "name.nii" not "name"
    # so warp files on disk are named {t1_stem}.nii_0GenericAffine.mat / _1Warp.nii.gz
    t1_warp_stem  = f"{t1_stem}.nii"
    affine_t1_mni = warps_dir / f"{t1_warp_stem}_0GenericAffine.mat"
    warp_t1_mni   = warps_dir / f"{t1_warp_stem}_1Warp.nii.gz"

    for p in [b0_path, mask_path, t1_path, md_path, affine_t1_mni, warp_t1_mni]:
        if not p.exists():
            return dti_stem, "fail", f"missing {p.name}"

    try:
        import tempfile as _tmp, os as _os

        fixed     = ants.image_read(str(t1_path))
        moving_b0 = ants.image_read(str(b0_path)) * ants.image_read(str(mask_path))

        md_ras = nib.as_closest_canonical(nib.load(str(md_path)))
        with _tmp.NamedTemporaryFile(suffix=".nii.gz", delete=False) as f:
            tmp = f.name
        nib.save(md_ras, tmp)
        moving_md = ants.image_read(tmp)
        _os.unlink(tmp)

        reg = ants.registration(
            fixed=fixed,
            moving=moving_b0,
            type_of_transform="Rigid",
            initial_transform="identity",
            aff_metric="mattes",
        )

        tx_saved = False
        for tx in reg["fwdtransforms"]:
            tx_p = Path(tx)
            if tx_p.suffix == ".mat" or "GenericAffine" in tx_p.name:
                shutil.copy(tx, out_tx)
                tx_saved = True
                break
        if not tx_saved:
            return dti_stem, "fail", "no .mat in fwdtransforms"

        md_in_t1 = ants.apply_transforms(
            fixed=fixed, moving=moving_md,
            transformlist=reg["fwdtransforms"],
            interpolator="linear",
        )
        ants.image_write(md_in_t1, str(out_md_t1))

        # Compose rigid DTI->T1 + SyN T1->MNI in one call.
        # Transform order: warp (output side) -> affine -> rigid (input side).
        md_in_mni = ants.apply_transforms(
            fixed=mni_t1,
            moving=moving_md,
            transformlist=[str(warp_t1_mni), str(affine_t1_mni), str(out_tx)],
            interpolator="linear",
        )
        ants.image_write(md_in_mni, str(out_md_mni))
        return dti_stem, "ok", ""

    except Exception as e:
        return dti_stem, "fail", str(e)


In [13]:
failed = []
skipped = 0
ok = 0

max_workers = max(1, (os.cpu_count() or 4) - 2)
print(f"Register b0→T1 + warp MD→MNI with {max_workers} workers ({len(md_maps)} subjects)")

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(_register_b0_and_warp_md, p) for p in md_maps]
    for fut in tqdm(as_completed(futures), total=len(md_maps), desc="b0→T1 + MD→MNI"):
        stem, status, err = fut.result()
        if status == "ok":
            ok += 1
        elif status == "skip":
            skipped += 1
        else:
            failed.append((stem, err))

mni_files = list(md_mni_dir.glob("*_MD.nii.gz"))
print(f"\nb0→T1 + MD→MNI complete:")
print(f"  {len(mni_files)} MD maps in MNI space")
print(f"  ({ok} new, {skipped} skipped, {len(failed)} failed)")
if failed:
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Register b0→T1 + warp MD→MNI with 14 workers (787 subjects)


b0→T1 + MD→MNI: 100%|██████████| 787/787 [43:20<00:00,  3.30s/it] 


b0→T1 + MD→MNI complete:
  786 MD maps in MNI space
  (786 new, 0 skipped, 1 failed)
  I1051717_006_S_4485a: no matching T1 for subject


In [14]:
# --- Smooth MNI MD (5 mm FWHM) -----------------------------------------------
fwhm_mm  = 5.0
sigma_mm = fwhm_mm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
mni_md_files = sorted(md_mni_dir.glob("*_MD.nii.gz"))
print(f"Smoothing {len(mni_md_files)} MD volumes @ {fwhm_mm} mm FWHM (sigma = {sigma_mm:.4f} mm)")

Smoothing 786 MD volumes @ 5.0 mm FWHM (sigma = 2.1233 mm)


In [15]:
def _smooth_md(md_mni_path):
    dti_stem = md_mni_path.name.replace("_MD.nii.gz", "")
    out_path = md_smooth_dir / f"{dti_stem}_MD_s5.nii.gz"
    if out_path.exists():
        return dti_stem, "skip", ""
    try:
        img = ants.image_read(str(md_mni_path))
        smoothed = ants.smooth_image(img, sigma=sigma_mm, sigma_in_physical_coordinates=True)
        # Scale mm2/s -> 10^-3 mm2/s (WM~0.7, GM~1.0, CSF~3.0)
        ants.image_write(smoothed * 1000.0, str(out_path))
        return dti_stem, "ok", ""
    except Exception as e:
        return dti_stem, "fail", str(e)


failed = []
skipped = ok = 0
max_workers = max(1, (os.cpu_count() or 4) - 2)

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(_smooth_md, p) for p in mni_md_files]
    for fut in tqdm(as_completed(futures), total=len(mni_md_files), desc="Smooth MD"):
        stem, status, err = fut.result()
        if status == "ok":
            ok += 1
        elif status == "skip":
            skipped += 1
        else:
            failed.append((stem, err))

smoothed_files = list(md_smooth_dir.glob("*_MD_s5.nii.gz"))
print(f"\nSmooth MD complete:")
print(f"  {len(smoothed_files)} smoothed MD maps")
print(f"  ({ok} new, {skipped} skipped, {len(failed)} failed)")
if failed:
    for name, err in failed[:10]:
        print(f"  {name}: {err}")

Smooth MD: 100%|██████████| 786/786 [03:38<00:00,  3.60it/s] 


Smooth MD complete:
  786 smoothed MD maps
  (786 new, 0 skipped, 0 failed)


### ML pre processing

In [21]:
# --- Build GM / WM masks resampled to the smoothed-image grid ----------------
# MNI tissue masks are 1 mm; smoothed PVEs are ~1.5 mm -> resample (NN).
# Use a T1 smoothed image as reference so this cell runs without DTI on disk.
import nilearn.image as nli

t1_smooth_dir = Path("model_data/adni/t1_long_data/t1_long_modulated_vbm/t1_long_pve_smoothed")
dti_smooth_dir = Path("model_data/adni/dti_long_data/dti_long_modulated_vbm/dti_long_mni_smoothed")
out_t1_dir = Path("model_data/adni/t1_long_data")
out_dti_dir = Path("model_data/adni/dti_long_data")
meta_csv = Path("model_data/adni/paired_df_long.csv")

# Prefer a T1 PVE file as reference; fall back to DTI if T1 not yet built
_t1_pve_candidates = sorted(t1_smooth_dir.glob("*/*_pve_1_mod_s2p5.nii.gz"))
_dti_candidates    = sorted(dti_smooth_dir.glob("*_MD_s5.nii.gz"))
if _t1_pve_candidates:
    ref_img = nib.load(str(_t1_pve_candidates[0]))
elif _dti_candidates:
    ref_img = nib.load(str(_dti_candidates[0]))
else:
    raise FileNotFoundError("No reference image found in T1 or DTI smooth dirs")

ref_source = "T1 PVE" if _t1_pve_candidates else "DTI MD"
ref_shape = ref_img.shape

def _load_mask(path):
    m = nli.resample_to_img(str(path), ref_img, interpolation="nearest", force_resample=True, copy_header=True)
    return np.asarray(m.dataobj) > 0.5

gm_mask  = _load_mask("model_data/mni_gm_mask_fast.nii")
wm_mask  = _load_mask("model_data/mni_wm_mask_fast.nii")
csf_mask = _load_mask("model_data/mni_csf_mask_fast.nii")
print(f"Reference grid ({ref_source}): {ref_shape}  |  GM voxels: {gm_mask.sum():,}  |  WM voxels: {wm_mask.sum():,}  |  CSF voxels: {csf_mask.sum():,}")

Reference grid (T1 PVE): (182, 218, 182)  |  GM voxels: 831,863  |  WM voxels: 785,285  |  CSF voxels: 337,620


In [23]:
# --- Build T1 parquets for the paired set ------------------------------------
# Load paired_stems from CSV if available (no DTI files needed on disk).
# Falls back to scanning DTI smooth dir if CSV doesn't exist yet.
from concurrent.futures import ThreadPoolExecutor

def _parse(stem):
    img, _, subj = stem.partition("_")
    return img.lstrip("I"), subj

if meta_csv.exists():
    _paired_csv = pd.read_csv(meta_csv)
    paired_stems = _paired_csv[["t1_image_subject_id", "dti_image_subject_id",
                                "t1_image_id", "dti_image_id", "subject_id"]]
    print(f"Loaded paired_stems from {meta_csv} ({len(paired_stems)} rows)")
else:
    t1_subjects = sorted([p for p in t1_smooth_dir.iterdir() if p.is_dir()])
    dti_files   = sorted(dti_smooth_dir.glob("*_MD_s5.nii.gz"))
    t1_all  = pd.DataFrame([(p.name, *_parse(p.name)) for p in t1_subjects],
                           columns=["t1_image_subject_id", "t1_image_id", "subject_id"])
    dti_all = pd.DataFrame([(s := f.name.replace("_MD_s5.nii.gz", ""), *_parse(s)) for f in dti_files],
                           columns=["dti_image_subject_id", "dti_image_id", "subject_id"])
    t1_all  = t1_all.sort_values(["subject_id", "t1_image_id"]).reset_index(drop=True)
    dti_all = dti_all.sort_values(["subject_id", "dti_image_id"]).reset_index(drop=True)
    t1_all["scan_n"]  = t1_all.groupby("subject_id").cumcount()
    dti_all["scan_n"] = dti_all.groupby("subject_id").cumcount()
    paired_stems = t1_all.merge(dti_all, on=["subject_id", "scan_n"], how="inner").drop(columns="scan_n")
    print(f"T1: {len(t1_all)}  DTI: {len(dti_all)}  Paired: {len(paired_stems)}")

# Filter to subjects whose smoothed PVE files are actually on disk
paired_t1_dirs = [
    d for s in paired_stems["t1_image_subject_id"]
    if (d := t1_smooth_dir / s) and (d / f"{s}_pve_1_mod_s2p5.nii.gz").exists()
]
n_missing = len(paired_stems) - len(paired_t1_dirs)
if n_missing:
    print(f"WARNING: {n_missing} paired subjects missing smoothed T1 files — skipped")
print(f"Extracting T1 parquets for {len(paired_t1_dirs)} subjects")

def _extract_t1(subj_dir):
    stem = subj_dir.name
    csf = nib.load(subj_dir / f"{stem}_pve_0_mod_s2p5.nii.gz").get_fdata(dtype=np.float32)
    gm  = nib.load(subj_dir / f"{stem}_pve_1_mod_s2p5.nii.gz").get_fdata(dtype=np.float32)
    wm  = nib.load(subj_dir / f"{stem}_pve_2_mod_s2p5.nii.gz").get_fdata(dtype=np.float32)
    return stem, gm[gm_mask], wm[wm_mask], csf[csf_mask]

stems, gm_rows, wm_rows, csf_rows = [], [], [], []
with ThreadPoolExecutor(max_workers=max(1, (os.cpu_count() or 4) - 2)) as ex:
    for stem, gm_vec, wm_vec, csf_vec in tqdm(
        ex.map(_extract_t1, paired_t1_dirs), total=len(paired_t1_dirs), desc="T1 mask -> parquet"
    ):
        stems.append(stem); gm_rows.append(gm_vec); wm_rows.append(wm_vec); csf_rows.append(csf_vec)

gm_df  = pd.DataFrame(np.vstack(gm_rows),  index=pd.Index(stems, name="image_subject_id"))
wm_df  = pd.DataFrame(np.vstack(wm_rows),  index=pd.Index(stems, name="image_subject_id"))
csf_df = pd.DataFrame(np.vstack(csf_rows), index=pd.Index(stems, name="image_subject_id"))
gm_df.columns  = gm_df.columns.astype(str)
wm_df.columns  = wm_df.columns.astype(str)
csf_df.columns = csf_df.columns.astype(str)
gm_df.to_parquet(out_t1_dir  / "t1_long_masked_gm.parquet",  compression="zstd")
wm_df.to_parquet(out_t1_dir  / "t1_long_masked_wm.parquet",  compression="zstd")
csf_df.to_parquet(out_t1_dir / "t1_long_masked_csf.parquet", compression="zstd")
print(f"Saved t1_long_masked_gm.parquet {gm_df.shape}, t1_long_masked_wm.parquet {wm_df.shape}, t1_long_masked_csf.parquet {csf_df.shape}")

Loaded paired_stems from model_data/adni/paired_df_long.csv (786 rows)
Extracting T1 parquets for 786 subjects


T1 mask -> parquet: 100%|██████████| 786/786 [00:25<00:00, 30.25it/s]


Saved t1_long_masked_gm.parquet (786, 831863), t1_long_masked_wm.parquet (786, 785285), t1_long_masked_csf.parquet (786, 337620)


In [24]:
# --- DTI MD -> masked GM-MD / WM-MD parquets (paired set only) ---------------
def _extract_md(md_path):
    stem = md_path.name.replace("_MD_s5.nii.gz", "")
    md = nib.load(md_path).get_fdata(dtype=np.float32)
    return stem, md[gm_mask], md[wm_mask], md[csf_mask]

paired_dti_files = [dti_smooth_dir / f"{s}_MD_s5.nii.gz" for s in paired_stems["dti_image_subject_id"]]
print(f"Building DTI parquets for {len(paired_dti_files)} paired subjects")

stems, gm_rows, wm_rows, csf_rows = [], [], [], []
with ThreadPoolExecutor(max_workers=max(1, (os.cpu_count() or 4) - 2)) as ex:
    for stem, gm_vec, wm_vec, csf_vec in tqdm(ex.map(_extract_md, paired_dti_files), total=len(paired_dti_files), desc="DTI mask -> parquet"):
        stems.append(stem); gm_rows.append(gm_vec); wm_rows.append(wm_vec); csf_rows.append(csf_vec)

dti_gm  = pd.DataFrame(np.vstack(gm_rows),  index=pd.Index(stems, name="image_subject_id"))
dti_wm  = pd.DataFrame(np.vstack(wm_rows),  index=pd.Index(stems, name="image_subject_id"))
dti_csf = pd.DataFrame(np.vstack(csf_rows), index=pd.Index(stems, name="image_subject_id"))
dti_gm.columns = dti_gm.columns.astype(str); dti_wm.columns = dti_wm.columns.astype(str); dti_csf.columns = dti_csf.columns.astype(str)
dti_gm.to_parquet(out_dti_dir  / "dti_long_masked_gm_md.parquet",  compression="zstd")
dti_wm.to_parquet(out_dti_dir  / "dti_long_masked_wm_md.parquet",  compression="zstd")
dti_csf.to_parquet(out_dti_dir / "dti_long_masked_csf_md.parquet", compression="zstd")
print(f"Saved dti_long_masked_gm_md.parquet {dti_gm.shape}, dti_long_masked_wm_md.parquet {dti_wm.shape}, dti_long_masked_csf_md.parquet {dti_csf.shape}")

Building DTI parquets for 786 paired subjects


DTI mask -> parquet: 100%|██████████| 786/786 [00:13<00:00, 56.97it/s]


Saved dti_long_masked_gm_md.parquet (786, 831863), dti_long_masked_wm_md.parquet (786, 785285), dti_long_masked_csf_md.parquet (786, 337620)


In [7]:
# ── Rebuild paired_df using dti_to_t1 (date-matched) + closest-date DX + amyloid ──
import re, pandas as pd, numpy as np
from pathlib import Path

# ── 1. Pair stems from the date-matched dti_to_t1 dict ───────────────────────
# dti_to_t1 already built in earlier cell: {dti_stem: t1_stem}
rows = []
for dti_stem, t1_stem in dti_to_t1.items():
    dti_img_id = dti_stem.split("_")[0].lstrip("I")
    t1_img_id  = t1_stem.replace(".nii", "").split("_")[0].lstrip("I")
    subj = "_".join(dti_stem.split("_")[1:])
    rows.append(dict(
        subject_id           = subj,
        t1_image_id          = t1_img_id,
        dti_image_id         = dti_img_id,
        t1_image_subject_id  = f"I{t1_img_id}_{subj}",
        dti_image_subject_id = f"I{dti_img_id}_{subj}",
    ))
paired_v2 = pd.DataFrame(rows)
print(f"Paired rows from dti_to_t1: {len(paired_v2)}  | unique subjects: {paired_v2['subject_id'].nunique()}")

# ── 2. T1 scan date from DCM folder ─────────────────────────────────────────
_dcm_root = Path("model_data/adni/t1_long_data/t1_long_dcm")
_date_pat  = re.compile(r"^(\d{4}-\d{2}-\d{2})")
_img_to_scandate = {}
for _sd in _dcm_root.iterdir():
    if not _sd.is_dir(): continue
    for _md in _sd.iterdir():
        if not _md.is_dir(): continue
        for _dd in _md.iterdir():
            _m = _date_pat.match(_dd.name)
            if not _m: continue
            _dt = pd.Timestamp(_m.group(1))
            for _id in _dd.iterdir():
                if _id.is_dir():
                    _img_to_scandate[_id.name.lstrip("I")] = _dt

paired_v2["scan_date"] = pd.to_datetime(paired_v2["t1_image_id"].map(_img_to_scandate))
print(f"Scan date missing: {paired_v2['scan_date'].isna().sum()} rows")

Paired rows from dti_to_t1: 786  | unique subjects: 528
Scan date missing: 0 rows


In [8]:
# ── 3. Diagnosis: closest DXSUM EXAMDATE within 365 days ─────────────────────
dxsum = pd.read_csv("model_data/adni/DXSUM_22Sep2025.csv",
                    usecols=["PTID", "EXAMDATE", "DIAGNOSIS"], low_memory=False)
dxsum["EXAMDATE"]  = pd.to_datetime(dxsum["EXAMDATE"],  errors="coerce")
dxsum["DIAGNOSIS"] = pd.to_numeric(dxsum["DIAGNOSIS"], errors="coerce")
dxsum = dxsum.dropna(subset=["EXAMDATE","DIAGNOSIS"]).loc[lambda z: z["DIAGNOSIS"].isin([1,2,3])]

def _closest_dx(row, max_days=365):
    sub = dxsum[dxsum["PTID"] == row["subject_id"]]
    if sub.empty or pd.isna(row["scan_date"]): return np.nan
    delta = (sub["EXAMDATE"] - row["scan_date"]).abs()
    if delta.min() > pd.Timedelta(days=max_days): return np.nan
    return int(sub.loc[delta.idxmin(), "DIAGNOSIS"])

paired_v2["_dx"] = paired_v2.apply(_closest_dx, axis=1)
paired_v2["group"]      = paired_v2["_dx"].map({1:"CN", 2:"MCI", 3:"Dementia"})
paired_v2["diag_label"] = paired_v2["_dx"].map({1:0, 2:1, 3:1})
paired_v2 = paired_v2.drop(columns="_dx")
print(f"group NaN (no DXSUM within 365d): {paired_v2['group'].isna().sum()}")

# ── 4. Amyloid: closest PET scan date to T1 scan date ────────────────────────
amy = pd.read_csv("model_data/adni/All_Subjects_UCBERKELEY_AMY_6MM_08Feb2026.csv",
                  usecols=["PTID","SCANDATE","AMYLOID_STATUS"], low_memory=False)
amy["SCANDATE"] = pd.to_datetime(amy["SCANDATE"], errors="coerce")
amy = amy.dropna(subset=["SCANDATE","AMYLOID_STATUS"])

def _closest_amy(row):
    sub = amy[amy["PTID"] == row["subject_id"]]
    if sub.empty or pd.isna(row["scan_date"]): return pd.NA
    idx = (sub["SCANDATE"] - row["scan_date"]).abs().idxmin()
    return int(sub.loc[idx, "AMYLOID_STATUS"])

paired_v2["amyloid_label"] = paired_v2.apply(_closest_amy, axis=1)

group NaN (no DXSUM within 365d): 16


In [10]:
# ── 5. Save ───────────────────────────────────────────────────────────────────
out_cols = ["subject_id","t1_image_id","dti_image_id",
            "t1_image_subject_id","dti_image_subject_id",
            "group","diag_label","amyloid_label"]
paired_v2[out_cols].to_csv("model_data/adni/paired_df_long_v2.csv", index=False)
print(f"\nSaved paired_df_long_v2.csv")


Saved paired_df_long_v2.csv


In [11]:
# ── 6. Compare v1 vs v2 ───────────────────────────────────────────────────────
v1 = pd.read_csv("model_data/adni/paired_df_long.csv")
v2 = paired_v2[out_cols].dropna(subset=["group"])

print(f"\n{'':30s} {'v1':>8} {'v2':>8}")
print(f"{'Rows':30s} {len(v1):>8} {len(v2):>8}")
print(f"{'Unique subjects':30s} {v1['subject_id'].nunique():>8} {v2['subject_id'].nunique():>8}")
for grp in ["CN","MCI","Dementia"]:
    print(f"  {grp:28s} {(v1['group']==grp).sum():>8} {(v2['group']==grp).sum():>8}")
print(f"{'amyloid+ (1)':30s} {(v1['amyloid_label']==1).sum():>8} {(v2['amyloid_label']==1).sum():>8}")
print(f"{'amyloid- (0)':30s} {(v1['amyloid_label']==0).sum():>8} {(v2['amyloid_label']==0).sum():>8}")
print(f"{'amyloid NaN':30s} {v1['amyloid_label'].isna().sum():>8} {v2['amyloid_label'].isna().sum():>8}")


                                     v1       v2
Rows                                786      770
Unique subjects                     528      520
  CN                                420      411
  MCI                               286      282
  Dementia                           80       77
amyloid+ (1)                        376      332
amyloid- (0)                        347      379
amyloid NaN                          63       59


In [13]:
# Subjects where group label changed
merged = v1.merge(v2[["t1_image_subject_id","group","amyloid_label"]],
                  on="t1_image_subject_id", suffixes=("_v1","_v2"), how="inner")
group_changed  = (merged["group_v1"]  != merged["group_v2"]).sum()
a1 = merged["amyloid_label_v1"].fillna(-1)
a2 = merged["amyloid_label_v2"].fillna(-1)
amyloid_changed = (a1 != a2).sum()
print(f"\nMatched rows (same t1_image_subject_id): {len(merged)}")
print(f"  group label changed:   {group_changed}")
print(f"  amyloid label changed: {amyloid_changed}")

if group_changed:
    print("\nGroup changes (first 10):")
    print(merged[merged["group_v1"] != merged["group_v2"]]
          [["subject_id","t1_image_subject_id","group_v1","group_v2"]].head(10).to_string(index=False))
if amyloid_changed:
    print("\nAmyloid changes (first 10):")
    mask = a1 != a2
    print(merged[mask]
          [["subject_id","t1_image_subject_id","amyloid_label_v1","amyloid_label_v2"]].head(10).to_string(index=False))


Matched rows (same t1_image_subject_id): 770
  group label changed:   0
  amyloid label changed: 60

Amyloid changes (first 10):
subject_id t1_image_subject_id  amyloid_label_v1 amyloid_label_v2
002_S_1280  I491256_002_S_1280               1.0                0
002_S_6103  I938767_002_S_6103               1.0                0
003_S_1122 I1020423_003_S_1122               1.0                0
003_S_1122 I1083056_003_S_1122               1.0                0
003_S_1122 I1162367_003_S_1122               1.0                0
003_S_1122  I853501_003_S_1122               1.0                0
003_S_4350 I1252848_003_S_4350               1.0                0
003_S_5154  I992566_003_S_5154               1.0                0
003_S_6256  I973278_003_S_6256               1.0                0
003_S_7010 I1495816_003_S_7010               1.0                0


/var/folders/4r/2w_vg8k91mldqsvxyw1y1cnml0bjbd/T/ipykernel_6304/2221446338.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  a2 = merged["amyloid_label_v2"].fillna(-1)
